# Explorando `alinet/spoken_squad` (Spoken-SQuAD)

Este notebook inspeciona o dataset [**alinet/spoken_squad**](https://huggingface.co/datasets/alinet/spoken_squad): formato no Hugging Face, *configs*, *splits*, *features* e as **5 primeiras** amostras de um subconjunto típico (WER44).

**Referência:** Lee et al., *Spoken SQuAD: A Study of Mitigating the Impact of Speech Recognition Errors on Listening Comprehension* (Interspeech 2018) — [arXiv:1804.00320](https://arxiv.org/abs/1804.00320).

**Ideia do dataset:** partindo do SQuAD, o campo `context` deixa de ser o parágrafo escrito original e passa a ser uma **transcrição simulada de fala** (com erros de ASR). A pergunta (`question`) permanece em texto; as respostas anotadas seguem o padrão SQuAD. Os subconjuntos **WER44** e **WER54** correspondem a diferentes níveis de taxa de erro de palavras (WER) na transcrição.

## Configurações (*configs*) e divisões (*splits*)

| Config    | Splits disponíveis   | Papel típico |
|-----------|----------------------|--------------|
| `default` | `train`, `validation` | Versão “completa” (~42k exemplos no *card* do Hub) |
| `WER44`   | `test`               | Teste com transcrição ASR (~WER 44%) |
| `WER54`   | `test`               | Teste com transcrição ASR (~WER 54%) |

No *viewer* do Hub costuma aparecer o split `test` com ~5,35k linhas para WER44/WER54 (número exato pode variar com revisões do repositório).

**Formato de armazenamento no Hub:** os ficheiros são servidos em **Parquet** (conversão automática), mas o esquema lógico é o de QA estilo SQuAD (JSON-like).

## Esquema de cada exemplo (*features*)

Cada linha é um dicionário com:

| Campo | Tipo | Significado |
|-------|------|-------------|
| `id` | `string` | Identificador SQuAD (liga o exemplo ao conjunto original) |
| `title` | `string` | Título do artigo / tópico (ex.: `Super_Bowl_50`) |
| `context` | `string` | **Passagem “falada”**: transcrição com ruído de ASR (é aqui que o dataset difere do SQuAD escrito) |
| `question` | `string` | Pergunta (texto, como no SQuAD) |
| `answers` | `dict` | `{"text": [str, ...], "answer_start": [int, ...]}` — até 3 anotações alternativas (iguais ao SQuAD) |

**Atenção (importante para modelos e métricas):** os valores em `answer_start` foram definidos para o **texto do parágrafo original do SQuAD**, não para a *string* `context` com erros de ASR. Por isso, em geral **`context[answer_start:answer_start+len(text)]` não coincide** com `answers["text"]`. Para avaliação em Spoken-SQuAD usa-se normalmente o **texto da resposta** (e/ou normalização), não o deslocamento no `context` ASR. A célula mais abaixo demonstra isso alinhando com o `squad` original pelo mesmo `id`.

In [1]:
# Requer: pip install datasets  (no projeto: uv pip install datasets)
from datasets import get_dataset_config_names, get_dataset_split_names, load_dataset

DATASET = "alinet/spoken_squad"
CONFIG = "WER44"  # experimente "WER54" ou carregue "default" com train/validation

configs = get_dataset_config_names(DATASET)
print("Configs:", configs)
for name in configs:
    print(f"  {name}: splits = {get_dataset_split_names(DATASET, name)}")

# Apenas as 5 primeiras linhas do split test (evita carregar o split inteiro na memória)
ds = load_dataset(DATASET, CONFIG, split="test[:5]")
print(f"\nCarregado: {DATASET} / {CONFIG!r}, split test[:5] → {len(ds)} exemplos")
print("Features:", ds.features)

/home/marcos_paulo/Documents/CLASP/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Configs: ['default', 'WER44', 'WER54']
  default: splits = ['train', 'validation']
  WER44: splits = ['test']
  WER54: splits = ['test']

Carregado: alinet/spoken_squad / 'WER44', split test[:5] → 5 exemplos
Features: {'id': Value('string'), 'title': Value('string'), 'context': Value('string'), 'question': Value('string'), 'answers': {'text': List(Value('string')), 'answer_start': List(Value('int64'))}}


In [2]:
import textwrap
import pandas as pd

rows = []
for i, ex in enumerate(ds):
    rows.append(
        {
            "i": i,
            "id": ex["id"],
            "title": ex["title"],
            "question": ex["question"],
            "n_answers": len(ex["answers"]["text"]),
            "answers_text": ex["answers"]["text"],
            "context_len": len(ex["context"]),
        }
    )

pd.set_option("display.max_colwidth", 120)
display(pd.DataFrame(rows))

,i,id,title,question,n_answers,answers_text,context_len
0,0,56be4db0acb8001400a502ec,Super_Bowl_50,Which NFL team represented the AFC at Super Bowl 50?,3,"[Denver Broncos, Denver Broncos, Denver Broncos]",819
1,1,56be4db0acb8001400a502ed,Super_Bowl_50,Which NFL team represented the NFC at Super Bowl 50?,3,"[Carolina Panthers, Carolina Panthers, Carolina Panthers]",819
2,2,56be4db0acb8001400a502ee,Super_Bowl_50,Where did Super Bowl 50 take place?,3,"[Santa Clara, California, Levi's Stadium, Levi's Stadium in the San Francisco Bay Area at Santa Clara, California.]",819
3,3,56be4db0acb8001400a502ef,Super_Bowl_50,Which NFL team won Super Bowl 50?,3,"[Denver Broncos, Denver Broncos, Denver Broncos]",819
4,4,56be4db0acb8001400a502f0,Super_Bowl_50,What color was used to emphasize the 50th anniversary of the Super Bowl?,3,"[gold, gold, gold]",819


In [3]:
def show_example(ex: dict, max_context_chars: int = 900) -> None:
    ctx = ex["context"]
    preview = ctx if len(ctx) <= max_context_chars else ctx[:max_context_chars] + "…"
    print("=" * 72)
    print("id:     ", ex["id"])
    print("title:  ", ex["title"])
    print("question:\n", textwrap.fill(ex["question"], width=100))
    print(f"\ncontext ({len(ctx)} chars), início:\n", textwrap.fill(preview, width=100))
    print("\nanswers:")
    for j, (t, s) in enumerate(zip(ex["answers"]["text"], ex["answers"]["answer_start"])):
        snippet = ctx[s : s + len(t)]
        ok = snippet == t
        print(f"  [{j}] text={t!r}")
        print(f"       answer_start={s} → slice in context matches text? {ok}")
        if not ok:
            print(f"       slice in context: {snippet!r}")


for ex in ds:
    show_example(ex)

id:      56be4db0acb8001400a502ec
title:   Super_Bowl_50
question:
 Which NFL team represented the AFC at Super Bowl 50?

context (819 chars), início:
 Favorable fifty with an american football game to determine the champion of the national football
leave an f for the twenty fifteen feet then. The american football conference fayette fi champion
denver broncos defeated the national football conference and effie champion carolina panthers twenty
four to ten to earn their parents super bowl title. The game was played on february seventh twenty
fifteen and revive stadium in the fan from fifth kobe area santa clara california. If if with the
fiftieth super bowl the league emphasize the golden anniversary with various goldstein's initiative
as well as temporarily defending the tradition of naming each super bowl game with roman numeral
find her with a game would have been known as favorable felt that the logo could prominently
featured the arabic numerals fifty.

answers:
  [0] text='Denver

## Alinhar `answer_start` com o SQuAD **escrito** (mesmo `id`)

Carrega apenas o exemplo necessário do `squad` para mostrar que `answer_start` aponta para o parágrafo **original**.

In [4]:
from datasets import load_dataset

spoken = ds[0]
sid = spoken["id"]

# validation do SQuAD contém sobreposição com perguntas de desenvolvimento; basta encontrar o mesmo id
squad_val = load_dataset("squad", split="validation")
squad_ex = next(row for row in squad_val if row["id"] == sid)

t0 = squad_ex["answers"]["text"][0]
s0 = squad_ex["answers"]["answer_start"][0]
slice_squad = squad_ex["context"][s0 : s0 + len(t0)]

print("Mesmo id:", sid)
print("Gold answer text[0]:", repr(t0))
print("SQuAD context slice at answer_start:", repr(slice_squad))
print("Match no parágrafo SQuAD original:", slice_squad == t0)
print("\nTrecho do contexto SPOKEN (ASR) onde aparece a resposta (minúsculas):")
needle = t0.lower()
ctx_spoken = spoken["context"].lower()
pos = ctx_spoken.find(needle)
print("  posição substring exata (casefold):", pos)
if pos >= 0:
    print("  …", spoken["context"][max(0, pos - 40) : pos + len(t0) + 40], "…")

Mesmo id: 56be4db0acb8001400a502ec
Gold answer text[0]: 'Denver Broncos'
SQuAD context slice at answer_start: 'Denver Broncos'
Match no parágrafo SQuAD original: True

Trecho do contexto SPOKEN (ASR) onde aparece a resposta (minúsculas):
  posição substring exata (casefold): 196
  … football conference fayette fi champion denver broncos defeated the national football conferen …


## Resumo

- **O que muda em relação ao SQuAD:** só o `context`, que simula **transcrição de fala com erros**; `question` e os textos em `answers["text"]` mantêm a intenção do SQuAD.
- **WER44 vs WER54:** dois níveis de degradação da transcrição (maior WER ⇒ contexto tipicamente mais difícil).
- **`default`:** traz `train` e `validation` para treino/ajuste; **WER44/WER54** trazem `test` para avaliação no regime “spoken”.
- **`answer_start`:** trata como metadado herdado do SQuAD sobre o **texto original**; para extrair evidência no `context` ASR, usa busca por texto normalizado, *span prediction* no `context`, ou outra estratégia adequada ao teu modelo.

Para carregar mais exemplos, troca `test[:5]` por `test`, `test[:100]`, etc.